# Fit quality and robustness
Code by Mieli Luukinen, licensed with the Academic Free License v3.0.

There are only few comments, with the main purpose of describing what needs to be done to replicate our results with your own data, rather than to explain what the code does. I have attempted to remove all code that isn't relevant to the final publication, but some remnants may still be there. There is also a slight risk that in that process I have removed something that is relevant, but I don't think so.

## Preparations

In [ ]:
%matplotlib widget

In [ ]:
# These supposedly limit the use of CPU cores, which may be useful on a shared server.

import os

NUM_THREADS = "16"

os.environ["OMP_NUM_THREADS"] = NUM_THREADS
os.environ["OPENBLAS_NUM_THREADS"] = NUM_THREADS
os.environ["MKL_NUM_THREADS"] = NUM_THREADS
os.environ["VECLIB_MAXIMUM_THREADS"] = NUM_THREADS
os.environ["NUMEXPR_NUM_THREADS"] = NUM_THREADS

In [ ]:
from os import listdir
from os.path import isfile, join
import csv
import datetime
import itertools
import json
import pickle

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.gridspec import GridSpec
import scipy.signal, scipy.stats, scipy.optimize, scipy.ndimage
from IPython.display import clear_output
from tqdm.notebook import tqdm, trange
from tqdm.contrib.concurrent import process_map
from tqdm import tqdm as tqdm_console
from statsmodels.stats.descriptivestats import sign_test
from statsmodels.discrete.discrete_model import Logit
from statsmodels.regression.mixed_linear_model import MixedLM
import openpyxl
import pandas
from tabulate import tabulate, SEPARATING_LINE
from docx import Document

from utils import *
from fit_analysis import *

In [ ]:
figfont8 = {'fontname':'Times New Roman', 'fontsize':8}
figfont10 = {'fontname':'Times New Roman', 'fontsize':10}
matplotlib.rcParams['mathtext.fontset'] = 'stix'

In [ ]:
# So, lots of data in different folders. You probably need to do some tweaking to use the code with your own data. Sadly, I don't have exact specifications for the formats our data are in, so you are
# mostly on your own. The main thing is that you need scored polysomnograms and psychomotor vigilance task data.

# Here the path for photoplethysmogram data is commented out due to some changes in our directories after I had extracted what I needed from them and saved it elsewhere. You should probably change that.
# EDIT: in the revision, I am again analysing the photoplethysmogram data for normalization and to assess the goodness of fit. For simplicity and saving RAM I will not re-do the whole thing where PPG
# data would be stored for every arousal. Thus, the comments about the need to rewrite some PPG handling code if you replicate the study are still valid.

event_path = '/wrk/luukinen/data/event_files'
header_path = '/wrk/luukinen/data/headers/'
hypno_path = '/wrk/luukinen/data/hypnograms'
pvt_path = '/wrk/luukinen/data/PVT_results'
headerstart = 'header_Compumedics ProFusion PSG - '
headerend = '.json'
eventstart = 'events_Compumedics ProFusion PSG - '
eventend = '.TXT'
hypnostart = 'hypnogram_Compumedics ProFusion PSG - '
hypnoend = '.TXT'
pvtstart = 'pvt_'
pvtend = '.xlsx'
demographics_path = '/wrk/luukinen/patientdata.xlsx'

pleth_path = '/wrk/luukinen/data/pa/pa_extracted_fixed/psg_2015_2017/'
plethend = 'Pleth_64Hz/data.npy'

In [ ]:
event_files = [f for f in tqdm(listdir(event_path)) if isfile(join(event_path, f))]
header_files = [f for f in tqdm(listdir(header_path)) if isfile(join(header_path, f))]
hypnograms = [f for f in tqdm(listdir(hypno_path)) if isfile(join(hypno_path, f))]
pleth_files = [f for f in tqdm(listdir(pleth_path)) if isfile(join(pleth_path, f, plethend))]
pvt_files = [f for f in tqdm(listdir(pvt_path)) if isfile(join(pvt_path, f))]
# A somewhat slow way of dealing with the fact that edf_files has a subfolder.
# It would be faster to sort the lists and discard the last one in that one.
# However, this is more robust if someone changes  the files.

# Those last comments are old, as seen from the fact the code no longer uses data in EDF format.

In [ ]:
fit_path = '/wrk/luukinen/results/20220426/'
patients_fit = [f for f in tqdm(listdir(fit_path)) if isfile(join(fit_path, f))]
fit_path_spont = '/wrk/luukinen/results/20221101_spont/'
patients_fit_spont = [f for f in tqdm(listdir(fit_path_spont)) if isfile(join(fit_path_spont, f))]

In [ ]:
# Matching files. The code is ugly due to some varying file name conventions etc.
files = {hdr.split()[4][1:]: [header_path + '/' + hdr,
 event_path + '/' + eventstart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + eventend,
 hypno_path + '/' + hypnostart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + hypnoend,
 pleth_path + hdr.split()[4][1:] + '/' + plethend,
 pvt_path + '/' + pvtstart + hdr.split()[4][1:] + pvtend]
 for hdr in tqdm(header_files)
 if (eventstart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + eventend) in event_files
 and (hypnostart + hdr.split()[4] + ' ' + hdr.split()[5][:-5] + hypnoend) in hypnograms
 and hdr.split()[4][1:] in pleth_files
 and pvtstart + hdr.split()[4][1:] + pvtend in pvt_files
 and (hdr.split()[4][1:] in patients_fit or hdr.split()[4][1:] in patients_fit_spont)
}

In [ ]:
patient_ids = sorted(list(files))

In [ ]:
wb = openpyxl.load_workbook('/wrk/luukinen/patientdata.xlsx', data_only=True)
ws = wb.active
demographics = []
for row in ws.values:
    demographics.append([])
    for value in row:
        demographics[-1].append(value)

In [ ]:
demographics_by_patient = {str(demographics[i][0]): demographics[i] for i in range(len(demographics)) if isinstance(demographics[i][0], int) and str(demographics[i][0]) in patient_ids}

In [ ]:
len([v for v in demographics_by_patient.values() if isinstance(v[3], int) or isinstance(v[3], float)])
exclude_no_demographics = []
for patient in patient_ids:
    v = demographics_by_patient[patient]
    if not (v[1] == 0 or v[1] == 1) or not (isinstance(v[2], int) or isinstance(v[2], float)) or not (isinstance(v[3], int) or isinstance(v[3], float)) or v[3] == 0:
        exclude_no_demographics.append(patient)
print(len(exclude_no_demographics))

# NOTE: This is untrue, check the number from previous paper!
# ANOTHER NOTE, MUCH LATER: This number is supposed to tell how many subjects were excluded for not having demographic data. I am no longer exactly sure why it's wrong.

In [ ]:
pvt_rts_by_patient = {}
for patient in tqdm(patient_ids):
    pvt_rts_by_patient[patient] = pandas.read_excel(files[patient][4])['rt'].to_numpy()

In [ ]:
# This data is desaturation severity (DesSev) for each subject, calculated elsewhere.

wb = openpyxl.load_workbook('/wrk/luukinen/DesSev_table.xlsx', data_only=True)
ws = wb.active
dessevs_ = []
for row in ws.values:
    dessevs_.append([])
    for value in row:
        dessevs_[-1].append(value)

In [ ]:
dessevs = {str(dessevs_[i][0]): dessevs_[i][1] for i in range(len(dessevs_)) if isinstance(dessevs_[i][0], int) and str(dessevs_[i][0]) in patient_ids}

In [ ]:
exclude_no_dessevs = []
for patient in patient_ids:
    if not patient in dessevs.keys() and not patient in exclude_no_demographics:
        exclude_no_dessevs.append(patient)
print(len(exclude_no_dessevs))

## Compiling arousal information

In [ ]:
n_patients = np.inf  # A maximum number. Actual amount may be less.
times = {}
durations = {}
stages = {}
stages2 = {}
epochs = {}
sats = {}
desats = {}
arousal_inds = {} # respiratory arousals taken into analyses
arousal_inds_resp_all = {}
arousal_inds_spont = {}
arousal_inds_all = {}
hypopnea_inds = {}
apnea_inds = {}
central_apnea_inds = {}
mixed_apnea_inds = {}
related_events = {}
pleths = {}

if n_patients < len(patient_ids):
    patients = list(np.random.choice(patient_ids, n_patients, replace=False))
else:
    patients = patient_ids.copy()
for patient in exclude_no_demographics:
    if patient in patients:
        patients.remove(patient)
for patient in exclude_no_dessevs:
    if patient in patients:
        patients.remove(patient)

for patient in tqdm(patients):
    filepath = files[patient][0]
    with open(filepath, newline='') as f:
        header = json.load(f)
    start = datetime.datetime.fromisoformat(header['startdate'])
    sample_rate_original = [signal_header for signal_header in header['SignalHeaders'] if signal_header['label'] == 'Pleth'][0]['sample_rate']
    filepath = files[patient][1]
    with open(filepath, newline='') as f:
        reader = csv.reader(f)
        events = list(reader)
    # ISO format:
    for event in events:
        if event[0][1] == ':':
            event[0] = '0'+event[0]
    filepath = files[patient][2]
    with open(filepath, newline='') as f:
        stages[patient] = [stage2num(str.strip('\r\n')) for str in f.readlines()]

    date = start.date()  # incorrect after midnight, but works for the substractions
    _times = [datetime.datetime.combine(date, datetime.time.fromisoformat(event[0])) for event in events]
    times[patient] = [(time-start).total_seconds() % (60*60*24) for time in _times]
    durations[patient] = [duration_from_str(event[4]) for event in events]
    stages2[patient] = [stage2num(event[2], ['AWAKE', 'REM', 'N1', 'N2', 'N3']) for event in events]
    epochs[patient] = [int(event[1])-1 for event in events]
    sats[patient] = [int(event[5]) if event[5] != '-' else 0 for event in events]
    desats[patient] = [int(event[6]) if event[6] != '-' else 0 for event in events]

    arousal_inds[patient] = [i for i in np.where([event[3] == 'Arousal 2 ARO RES' for event in events])[0] if 3 < durations[patient][i] < 15 and times[patient][i] + durations[patient][i] < len(stages[patient])*30]
    arousal_inds_resp_all[patient] = [i for i in np.where([event[3] == 'Arousal 2 ARO RES' for event in events])[0]]
    arousal_inds_spont[patient] = [i for i in np.where([event[3] == 'Arousal 1 ARO SPONT' for event in events])[0] if 3 < durations[patient][i] < 15 and times[patient][i] + durations[patient][i] < len(stages[patient])*30]
    arousal_inds_all[patient] = [i for i in np.where([event[3][0:7] == 'Arousal' for event in events])[0]]
    hypopnea_inds[patient] = np.where([event[3] == 'Hypopnea' for event in events])[0]
    apnea_inds[patient] = np.where([event[3] == 'Obstructive Apnea' for event in events])[0]
    central_apnea_inds[patient] = np.where([event[3] == 'Central Apnea' for event in events])[0]
    mixed_apnea_inds[patient] = np.where([event[3] == 'Mixed Apnea' for event in events])[0]
    related_events[patient] = []
    for i in arousal_inds[patient]:
        e = []
        for j in hypopnea_inds[patient]:
            if times[patient][i] - 5 < times[patient][j] + durations[patient][j] < times[patient][i] + durations[patient][i]:
                e.append(j)
        for j in apnea_inds[patient]:
            if times[patient][i] - 5 < times[patient][j] + durations[patient][j] < times[patient][i] + durations[patient][i]:
                e.append(j)
        related_events[patient].append(sorted(e))
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(arousal_inds[patient])) if len(related_events[patient][i]) == 1]
    related_events[patient] = [e[0] for e in related_events[patient] if len(e) == 1]
    for i in arousal_inds[patient]:
        if stages2[patient][i] > -1:
            stages2[patient][i] = stages[patient][epochs[patient][i]-1]
    for i in arousal_inds_spont[patient]:
        if stages2[patient][i] > -1:
            stages2[patient][i] = stages[patient][epochs[patient][i]-1]
    related_events[patient] = [related_events[patient][i] for i in range(len(related_events[patient])) if stages2[patient][arousal_inds[patient][i]] < 0]
    arousal_inds[patient] = [i for i in arousal_inds[patient] if stages2[patient][i] < 0]
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(arousal_inds[patient])) if 0 <= sats[patient][related_events[patient][i]] <= 100]
    related_events[patient] = [e for e in related_events[patient] if 0 <= sats[patient][e] <= 100]
    arousal_inds_spont[patient] = [i for i in arousal_inds_spont[patient] if stages2[patient][i] < 0]
    arousal_inds[patient] = [arousal_inds[patient][i] for i in range(len(related_events[patient])) if durations[patient][related_events[patient][i]] > 10]
    related_events[patient] = [i for i in related_events[patient] if durations[patient][i] > 10]
    # When replicating the study, add something like the following line.
    # pleths[patient] = scipy.signal.decimate(np.load(files[patient][3]), int(sample_rate_original/sample_rate))

In [ ]:
arousals_with_5s5s_margins = {}
arousals_with_10s5s_margins = {}
arousals_with_5s10s_margins = {}
arousals_with_10s10s_margins = {}
for patient in tqdm(patients):
    arousals_with_5s5s_margins[patient] = []
    arousals_with_10s5s_margins[patient] = []
    arousals_with_5s10s_margins[patient] = []
    arousals_with_10s10s_margins[patient] = []
    for i in range(len(arousal_inds[patient])):
        start = times[patient][arousal_inds[patient][i]]
        end = start + durations[patient][arousal_inds[patient][i]]
        within55 = False
        within105 = False
        within510 = False
        within1010 = False
        for j in np.concatenate((arousal_inds_all[patient], apnea_inds[patient], hypopnea_inds[patient], central_apnea_inds[patient], mixed_apnea_inds[patient])):
            if j != arousal_inds[patient][i] and j != related_events[patient][i]:
                if start - 5 < times[patient][j] < end + 5 or start - 5 < times[patient][j] + durations[patient][j] < end + 5:
                    within55 = True
                if start - 10 < times[patient][j] < end + 5 or start - 10 < times[patient][j] + durations[patient][j] < end + 5:
                    within105 = True
                if start - 5 < times[patient][j] < end + 10 or start - 5 < times[patient][j] + durations[patient][j] < end + 10:
                    within510 = True
                if start - 10 < times[patient][j] < end + 10 or start - 10 < times[patient][j] + durations[patient][j] < end + 10:
                    within1010 = True
        if not within55:
            arousals_with_5s5s_margins[patient].append(i)
        if not within105:
            arousals_with_10s5s_margins[patient].append(i)
        if not within510:
            arousals_with_5s10s_margins[patient].append(i)
        if not within1010:
            arousals_with_10s10s_margins[patient].append(i)
print(sum([len(arousal_inds[patient]) for patient in patients]))
print(sum([len(arousals_with_5s5s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_10s5s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_5s10s_margins[patient]) for patient in patients]))
print(sum([len(arousals_with_10s10s_margins[patient]) for patient in patients]))

In [ ]:
arousals_with_10s10s_margins_spont = {}
for patient in tqdm(patients):
    arousals_with_10s10s_margins_spont[patient] = []
    for i in range(len(arousal_inds_spont[patient])):
        start = times[patient][arousal_inds_spont[patient][i]]
        end = start + durations[patient][arousal_inds_spont[patient][i]]
        within1010 = False
        for j in np.concatenate((arousal_inds_all[patient], apnea_inds[patient], hypopnea_inds[patient], central_apnea_inds[patient], mixed_apnea_inds[patient])):
            if j != arousal_inds_spont[patient][i]:
                if start - 10 < times[patient][j] < end + 10 or start - 10 < times[patient][j] + durations[patient][j] < end + 10:
                    within1010 = True
        if not within1010:
            arousals_with_10s10s_margins_spont[patient].append(i)
print(sum([len(arousal_inds_spont[patient]) for patient in patients]))
print(sum([len(arousals_with_10s10s_margins_spont[patient]) for patient in patients]))

In [ ]:
print(len(patients))
patients = [patient for patient in patients if len(arousals_with_10s10s_margins[patient]) > 0 or len(arousals_with_10s10s_margins_spont[patient]) > 0]
print(len(patients))

In [ ]:
# A quick and lazy way of noting that one patient originally included is missing PPG data at the revision stage when the data has been brought back through a different process.
# No reason to check everything again for a couple dozen arousals in a set of tens of thousands, just need to update the numbers. For replicating the study this doesn't matter.
print(856 - len(patients))

## Fitting model quality analysis

In [ ]:
n_gof_patients = 100

test_types = [
    ('noise', 0),
    ('noise', 5),
    ('noise', 10),
    ('noise', 15),
    ('noise', 20)
]

gof_by_noise = {}
for ttype in test_types:
    if ttype[0] == 'noise':
        gof_by_noise[ttype[1]] = {}
        gof_by_noise[ttype[1]]['rmspe_per_std_freq'] = []
        gof_by_noise[ttype[1]]['rmspe_per_std_amp'] = []
        gof_by_noise[ttype[1]]['rmspe_per_mean_freq'] = []
        gof_by_noise[ttype[1]]['rmspe_per_mean_amp'] = []
        gof_by_noise[ttype[1]]['gof_popts_freq'] = {}
        gof_by_noise[ttype[1]]['gof_popts_amp'] = {}
        gof_by_noise[ttype[1]]['gof_pleths'] = {}
        gof_by_noise[ttype[1]]['gof_ydata_freq'] = {}
        gof_by_noise[ttype[1]]['gof_ydata_amp'] = {}
        gof_by_noise[ttype[1]]['gof_xdata_freq'] = {}
        gof_by_noise[ttype[1]]['gof_xdata_amp'] = {}

gof_patients = [patients[i] for i in np.random.choice(list(range(len(patients))), n_gof_patients, replace=False)]
gof_arousals = {}
for i, patient in tqdm(enumerate(gof_patients)):
    while not patient in arousals_with_10s10s_margins or not arousals_with_10s10s_margins[patient] or not isfile(join(pleth_path, patient, plethend)):
        while patient in gof_patients:
            patient = patients[np.random.choice(list(range(len(patients))))]
    gof_patients[i] = patient

    pleth = np.load(join(pleth_path, patient, plethend))
    pleth_time = np.arange(len(pleth)) / sample_rate
    arousal_pleth_inds = [np.where((times[patient][i] < pleth_time) & (pleth_time < times[patient][i] + durations[patient][i]))[0] for i in arousal_inds[patient]]
    arousal_i = np.random.choice(arousals_with_10s10s_margins[patient])
    gof_arousals[patient] = arousal_i
    inds = arousal_pleth_inds[arousal_i]
    inds = np.arange(min(inds)-sample_rate*before_arousal, max(inds)+1+sample_rate*before_arousal)
    while not (inds[-1] < len(pleth) and inds[0] > 0) or np.std(pleth[inds]) == 0:
        arousal_i = np.random.choice(arousals_with_10s10s_margins[patient])
        gof_arousals[patient] = arousal_i
        inds = arousal_pleth_inds[arousal_i]
        inds = np.arange(min(inds)-sample_rate*before_arousal, max(inds)+1+sample_rate*before_arousal)
    for noise in gof_by_noise:
        gof_by_noise[noise]['gof_pleths'][patient] = pleth[inds] + np.random.randn(len(inds)) * (noise / 100) * (np.max(pleth[inds]) - np.min(pleth[inds]))
        f, t, Sxx = scipy.signal.spectrogram(gof_by_noise[noise]['gof_pleths'][patient], fs=sample_rate, nperseg=sample_rate*2, nfft=sample_rate*16, noverlap=sample_rate*2-1, window='boxcar')
        t = t - before_arousal

        ydata_ = np.average(np.tile(f, (len(t), 1)).transpose(), axis=0, weights=Sxx)
        ydata = scipy.signal.savgol_filter(ydata_, 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
        xdata = t[sample_rate:-sample_rate+1]
        scale = max(ydata)
        gof_by_noise[noise]['gof_ydata_freq'][patient] = ydata
        gof_by_noise[noise]['gof_xdata_freq'][patient] = xdata

        try:
            popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
        except:
            popt = [np.nan]*7
        for startguess in [-5, 0, 5]:
            for midguess in [0.3, 0.5, 0.7]:
                for endguess in [0.5, 0.7, 0.9]:
                    try:
                        _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                        if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                            popt = _popt
                    except:
                        pass
        if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0 or np.nan in popt:
            print('unsuccessful freq fit!')
        else:
            gof_by_noise[noise]['gof_popts_freq'][patient] = popt
            ydata_fit = n_shape_margins(xdata, *popt)
            rmspe = np.sqrt(np.mean((ydata - ydata_fit)**2))
            gof_by_noise[noise]['rmspe_per_std_freq'].append(rmspe / np.std(ydata, ddof=1))
            gof_by_noise[noise]['rmspe_per_mean_freq'].append(rmspe / np.mean(ydata))

        ydata_ = scipy.ndimage.maximum_filter(gof_by_noise[noise]['gof_pleths'][patient], size=sample_rate*2) - scipy.ndimage.minimum_filter(gof_by_noise[noise]['gof_pleths'][patient], size=sample_rate*2)
        ydata = scipy.signal.savgol_filter(ydata_[sample_rate:-sample_rate+1], 2*sample_rate+1, polyorder=0)[sample_rate:-sample_rate+1]
        xdata = t[sample_rate:-sample_rate+1]
        scale = max(ydata)
        gof_by_noise[noise]['gof_ydata_amp'][patient] = ydata
        gof_by_noise[noise]['gof_xdata_amp'][patient] = xdata

        try:
            popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, -5, 0.5,  0.9])
        except:
            popt = [np.nan]*7
        for startguess in [-5, 0, 5]:
            for midguess in [0.3, 0.5, 0.7]:
                for endguess in [0.5, 0.7, 0.9]:
                    try:
                        _popt, _ = scipy.optimize.curve_fit(n_shape_margins, xdata, ydata, bounds=([0, -scale, 0, 0, min(xdata)+1, 0, 0],[scale, scale, np.inf, np.inf, max(xdata)-3, 1, 1]), p0=[np.median(ydata[:(before_arousal-2)*sample_rate]), 0, 1, 1, startguess, midguess, endguess])
                        if sum((ydata - n_shape_margins(xdata, *_popt))**2) < sum((ydata - n_shape_margins(xdata, *popt))**2) or np.any(np.isnan(popt)):
                            popt = _popt
                    except:
                        pass
        if popt[0] < 0 or popt[0] + popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] < 0 or popt[0] + popt[1] - popt[2]*popt[1] + popt[3]*popt[1] < 0:
            print('unsuccessful amp fit!')
        else:
            gof_by_noise[noise]['gof_popts_amp'][patient] = popt
            ydata_fit = n_shape_margins(xdata, *popt)
            rmspe = np.sqrt(np.mean((ydata - ydata_fit)**2))
            gof_by_noise[noise]['rmspe_per_std_amp'].append(rmspe / np.std(ydata, ddof=1))
            gof_by_noise[noise]['rmspe_per_mean_amp'].append(rmspe / np.mean(ydata))

with open('/wrk/luukinen/results/20260712_gof/gof.pickle', 'wb') as f:
    pickle.dump((gof_patients, gof_arousals, gof_by_noise), f)

In [ ]:
with open('/wrk/luukinen/results/20260712_gof/gof.pickle', 'rb') as f:
    gof_patients, gof_arousals, gof_by_noise = pickle.load(f)

In [ ]:
np.median(gof_by_noise[0]['rmspe_per_std_freq'])

In [ ]:
np.median(gof_by_noise[0]['rmspe_per_mean_freq'])

In [ ]:
np.median(gof_by_noise[0]['rmspe_per_std_amp'])

In [ ]:
np.median(gof_by_noise[0]['rmspe_per_mean_amp'])

In [ ]:
if 4922 in plt.get_fignums(): plt.close(4922)
fig= plt.figure(num=4922, figsize=(17/2.54, 6), dpi=100)
gs = GridSpec(5, 4)
ax = []
random_gof_patients1 = np.random.choice(gof_patients, 20, replace=False)
for i, patient in enumerate(random_gof_patients1):
    ax.append(fig.add_subplot(gs[i // 4, i % 4]))
    ax[-1].plot(gof_by_noise[0]['gof_xdata_freq'][patient], gof_by_noise[0]['gof_ydata_freq'][patient])
    ax[-1].plot(gof_by_noise[0]['gof_xdata_freq'][patient], n_shape_margins(gof_by_noise[0]['gof_xdata_freq'][patient], *gof_by_noise[0]['gof_popts_freq'][patient]))
    ax[-1].axvline(color='k')
    ax[-1].set_xticks([])
    ax[-1].set_yticks([])

ax[16].set_xlabel('Time', **figfont10)
ax[17].set_xlabel('Time', **figfont10)
ax[18].set_xlabel('Time', **figfont10)
ax[19].set_xlabel('Time', **figfont10)

ax[0].set_ylabel('Frequency', **figfont10)
ax[4].set_ylabel('Frequency', **figfont10)
ax[8].set_ylabel('Frequency', **figfont10)
ax[12].set_ylabel('Frequency', **figfont10)
ax[16].set_ylabel('Frequency', **figfont10)

plt.tight_layout()
plt.show()

In [ ]:
if 8603 in plt.get_fignums(): plt.close(8603)
fig= plt.figure(num=8603, figsize=(17/2.54, 6), dpi=100)
gs = GridSpec(5, 4)
ax = []
random_gof_patients2 = np.random.choice(gof_patients, 20, replace=False)
for i, patient in enumerate(random_gof_patients2):
    ax.append(fig.add_subplot(gs[i // 4, i % 4]))
    ax[-1].plot(gof_by_noise[0]['gof_xdata_amp'][patient], gof_by_noise[0]['gof_ydata_amp'][patient])
    ax[-1].plot(gof_by_noise[0]['gof_xdata_amp'][patient], n_shape_margins(gof_by_noise[0]['gof_xdata_amp'][patient], *gof_by_noise[0]['gof_popts_amp'][patient]))
    ax[-1].axvline(color='k')
    ax[-1].set_xticks([])
    ax[-1].set_yticks([])

ax[16].set_xlabel('Time', **figfont10)
ax[17].set_xlabel('Time', **figfont10)
ax[18].set_xlabel('Time', **figfont10)
ax[19].set_xlabel('Time', **figfont10)

ax[0].set_ylabel('Amplitude', **figfont10)
ax[4].set_ylabel('Amplitude', **figfont10)
ax[8].set_ylabel('Amplitude', **figfont10)
ax[12].set_ylabel('Amplitude', **figfont10)
ax[16].set_ylabel('Amplitude', **figfont10)

plt.tight_layout()
plt.show()

In [ ]:
for noiselevel in gof_by_noise:
    gof_by_noise[noiselevel]['freq_magnitudes'] = []
    gof_by_noise[noiselevel]['freq_delays'] = []
    gof_by_noise[noiselevel]['amp_magnitudes'] = []
    gof_by_noise[noiselevel]['amp_delays'] = []
    for patient in gof_patients:
        amp_fit = gof_by_noise[noiselevel]['gof_popts_amp'][patient]
        freq_fit = gof_by_noise[noiselevel]['gof_popts_amp'][patient]
        duration = durations[patient][arousal_inds[patient][gof_arousals[patient]]]
        amp_classification = classify(amp_fit, False, duration)
        amp_startbaseline = startbaseline(amp_fit, duration, amp_classification)
        amp_endbaseline = endbaseline(amp_fit, duration, amp_classification)
        amp_expected_change = expected_change(amp_fit, duration, amp_classification)
        amp_unexpected_change = unexpected_change(amp_fit, duration, amp_classification)
        amp_second_expected_change = second_expected_change(amp_fit, duration, amp_classification)
        amp_second_unexpected_change = second_unexpected_change(amp_fit, duration, amp_classification)
        amp_meanfitvalue = meanfitvalue(amp_fit, duration)
        if (not np.isnan(amp_second_expected_change[0])) and abs(amp_second_expected_change[0] - amp_startbaseline)*(duration+before_arousal-amp_second_expected_change[1]) > abs(amp_expected_change[0] - amp_startbaseline)*(amp_expected_change[2]-amp_expected_change[1]):
            amp_eventual_expected_change = amp_second_expected_change
        else:
            amp_eventual_expected_change = amp_expected_change
        gof_by_noise[noiselevel]['amp_magnitudes'].append(-amp_eventual_expected_change[0] + amp_startbaseline if not np.isnan(amp_eventual_expected_change[0]) else 0 if not np.isnan(amp_startbaseline) else np.nan)
        gof_by_noise[noiselevel]['amp_delays'].append(amp_eventual_expected_change[1])
        freq_classification = classify(freq_fit, True, duration)
        freq_startbaseline = startbaseline(freq_fit, duration, freq_classification)
        freq_endbaseline = endbaseline(freq_fit, duration, freq_classification)
        freq_expected_change = expected_change(freq_fit, duration, freq_classification)
        freq_unexpected_change = unexpected_change(freq_fit, duration, freq_classification)
        freq_second_expected_change = second_expected_change(freq_fit, duration, freq_classification)
        freq_second_unexpected_change = second_unexpected_change(freq_fit, duration, freq_classification)
        freq_meanfitvalue = meanfitvalue(freq_fit, duration)
        if (not np.isnan(freq_second_expected_change[0])) and abs(freq_second_expected_change[0] - freq_startbaseline)*(duration+before_arousal-freq_second_expected_change[1]) > abs(freq_expected_change[0] - freq_startbaseline)*(freq_expected_change[2]-freq_expected_change[1]):
            freq_eventual_expected_change = freq_second_expected_change
        else:
            freq_eventual_expected_change = freq_expected_change
        gof_by_noise[noiselevel]['freq_magnitudes'].append(freq_eventual_expected_change[0] - freq_startbaseline if not np.isnan(freq_eventual_expected_change[0]) else 0 if not np.isnan(freq_startbaseline) else np.nan)
        gof_by_noise[noiselevel]['freq_delays'].append(freq_eventual_expected_change[1])

In [ ]:
if 3961 in plt.get_fignums(): plt.close(3961)
fig= plt.figure(num=3961, figsize=(17/2.54, 6), dpi=100)
gs = GridSpec(4, 4)
ax = []
i = 0

for noiselevel in gof_by_noise:
    if noiselevel > 0:
        ax.append(fig.add_subplot(gs[i, 0]))
        ax[-1].scatter(gof_by_noise[0]['freq_magnitudes'], gof_by_noise[noiselevel]['freq_magnitudes'], marker='.')
        ax[-1].text(0.6, 0.3, f"$r$ = {scipy.stats.spearmanr(gof_by_noise[0]['freq_magnitudes'], gof_by_noise[noiselevel]['freq_magnitudes']).correlation:.4f}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].set_xlabel(f'({"abcdefghijklmnop"[4*i]})', **figfont8)
        ax.append(fig.add_subplot(gs[i, 1]))
        x = []
        y = []
        fp = 0
        fn = 0
        for j in range(len(gof_by_noise[0]['freq_delays'])):
            if not np.isnan(gof_by_noise[0]['freq_delays'][j]) and not np.isnan(gof_by_noise[noiselevel]['freq_delays'][j]):
                x.append(gof_by_noise[0]['freq_delays'][j])
                y.append(gof_by_noise[noiselevel]['freq_delays'][j])
            elif np.isnan(gof_by_noise[0]['freq_delays'][j]) and not np.isnan(gof_by_noise[noiselevel]['freq_delays'][j]):
                fp += 1
            elif not np.isnan(gof_by_noise[0]['freq_delays'][j]) and np.isnan(gof_by_noise[noiselevel]['freq_delays'][j]):
                fn += 1
        ax[-1].scatter(x,y, marker='.')
        ax[-1].text(0.6, 0.3, f"$r$ = {scipy.stats.spearmanr(x, y).correlation:.4f}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].text(0.6, 0.2, f"$FP$ = {fp}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].text(0.6, 0.1, f"$FN$ = {fn}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].set_xlabel(f'({"abcdefghijklmnop"[4*i+1]})', **figfont8)
        ax.append(fig.add_subplot(gs[i, 2]))
        ax[-1].scatter(gof_by_noise[0]['amp_magnitudes'], gof_by_noise[noiselevel]['amp_magnitudes'], marker='.')
        ax[-1].text(0.6, 0.3, f"$r$ = {scipy.stats.spearmanr(gof_by_noise[0]['amp_magnitudes'], gof_by_noise[noiselevel]['amp_magnitudes']).correlation:.4f}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].set_xlabel(f'({"abcdefghijklmnop"[4*i+2]})', **figfont8)
        ax.append(fig.add_subplot(gs[i, 3]))
        x = []
        y = []
        fp = 0
        fn = 0
        for j in range(len(gof_by_noise[0]['amp_delays'])):
            if not np.isnan(gof_by_noise[0]['amp_delays'][j]) and not np.isnan(gof_by_noise[noiselevel]['amp_delays'][j]):
                x.append(gof_by_noise[0]['amp_delays'][j])
                y.append(gof_by_noise[noiselevel]['amp_delays'][j])
            elif np.isnan(gof_by_noise[0]['amp_delays'][j]) and not np.isnan(gof_by_noise[noiselevel]['amp_delays'][j]):
                fp += 1
            elif not np.isnan(gof_by_noise[0]['amp_delays'][j]) and np.isnan(gof_by_noise[noiselevel]['amp_delays'][j]):
                fn += 1
        ax[-1].scatter(x, y, marker='.')
        ax[-1].text(0.6, 0.3, f"$r$ = {scipy.stats.spearmanr(x, y).correlation:.4f}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].text(0.6, 0.2, f"$FP$ = {fp}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].text(0.6, 0.1, f"$FN$ = {fn}", va='center', ha='left', transform=ax[-1].transAxes, **figfont8)
        ax[-1].set_xlabel(f'({"abcdefghijklmnop"[4*i+3]})', **figfont8)
        i += 1

ax[0].set_title('Freq. incr.', **figfont10)
ax[1].set_title('Freq. timing', **figfont10)
ax[2].set_title('Ampl. decr.', **figfont10)
ax[3].set_title('Ampl. timing', **figfont10)

ax[0].set_ylabel('Noise level:\n5% of SD', **figfont10)
ax[4].set_ylabel('Noise level:\n10% of SD', **figfont10)
ax[8].set_ylabel('Noise level:\n15% of SD', **figfont10)
ax[12].set_ylabel('Noise level:\n20% of SD', **figfont10)

for i in range(16):
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.tight_layout()
plt.show()